## Explore Dataset

In [ ]:
import pandas as pd
import requests as rq
import numpy as np
import geopandas as gpd
import json
import matplotlib.pyplot as plt
import shapely
import dotenv
import os

In [ ]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [ ]:
geo_JSON_url = "https://services2.arcgis.com/g6o32ZDQ33GpCIu3/arcgis/rest/services/Thames_Water_Storm_Overflow_Activity_(Production)_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso = gpd.read_file(geo_JSON_url)

In [ ]:
cso = cso.to_crs(epsg=3857)

In [ ]:
cso.head()

### River data

In [ ]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs(epsg=3857)

In [ ]:
rivers.crs

## Start Plotting geometry

### CSO Data

In [ ]:
cso.explore()

### River Data

In [ ]:
m = rivers[rivers.name1.str.contains("Thames",case=False,na=False)].explore()
cso.explore(m=m)

## Thames analysis

The idea is to try to find CSOs near the river thames and correctly associate them to the river. Then, it would be to assign a simple risk to the river based on the distance of the point from the CSO

### Analysis near the thames river

In [ ]:
cso_thames = cso.copy()
cso_thames["isClose"] = False

In [ ]:
river_thames = rivers[rivers["name1"] == "River Thames"].copy()
river_thames = river_thames.reset_index().drop(columns="index")

In [ ]:
# Dividing each river segment into equally spaced points

division_distance = 125 # metres.
river_thames["geometry2"] = river_thames.geometry.apply(
    lambda x: shapely.line_interpolate_point(
        x,
        np.arange(0,x.length,division_distance)
        # np.linspace(0,1,n_divisions),
        # normalized=True,
    )
)

In [ ]:
river_thames_expl = river_thames.explode("geometry2")

In [ ]:
river_thames_expl.geometry2 = river_thames_expl.geometry2.astype("geometry")

In [ ]:
river_thames_expl.geometry2 = river_thames_expl.geometry2.set_crs(river_thames.crs)

In [ ]:
river_thames_points = river_thames_expl.set_geometry("geometry2")

In [ ]:
river_thames_points

In [ ]:
river_to_explore = 136853
m = river_thames_points[river_thames_points["OBJECTID"]==river_to_explore].explore()
river_thames[river_thames["OBJECTID"]==river_to_explore].explore(m=m)

In [ ]:
river_thames_points.explore()

#### METHOD 1 : Using buffer distance

Caveats:
- The distance calculations do not have information about the coordinate system (Latitude and longitude) so if being done near the poles, they will be wildly off

In [ ]:
def match_points_to_lines(points_gdf: gpd.GeoDataFrame, 
                         lines_gdf: gpd.GeoDataFrame, 
                         max_distance_meters: float) -> gpd.GeoDataFrame:
    """
    Match points to nearest lines within distance using buffer approach.
    
    Args:
        points_gdf: GeoDataFrame with Point geometries in EPSG:4326
        lines_gdf: GeoDataFrame with LineString geometries in EPSG:4326
        max_distance_meters: Maximum distance threshold in meters
    """
    # Convert to projected CRS for accurate buffering
    points_projected = points_gdf.to_crs(epsg=3857)
    lines_projected = lines_gdf.to_crs(epsg=3857)
    
    # Buffer lines by max distance
    buffered_lines = lines_projected.copy()
    buffered_lines.geometry = lines_projected.geometry.buffer(max_distance_meters)
    
    # Spatial join points with buffered lines
    matched_points = gpd.sjoin_nearest(
        points_projected,
        buffered_lines,
        max_distance=max_distance_meters,
        distance_col="distance_meters"
    )
    
    return matched_points.to_crs()

In [ ]:
bounds_distance = 1E-3 # In terms of latitude and longitude offset
river_thames_bounds = river_thames.copy()
river_thames_bounds["bounds"] = river_thames.geometry.apply(lambda x: x.buffer(bounds_distance))

In [ ]:
river_thames_bounds.bounds = river_thames_bounds